# Frontier Models: API and Abstraction-Layer Reference

This notebook compares hosted and local LLMs through several interfaces: the OpenAI-compatible API, provider-native SDKs, OpenRouter, LangChain, LiteLLM, and Ollama. Use it as a copy-paste reference for client setup, request shapes, and how each wrapper returns text.

## Core request pattern

1. Load credentials from environment variables.
2. Initialize a provider client.
3. Build a list of `role`/`content` messages.
4. Send those messages to a selected model.
5. Extract and display the model's text response.

## Contents

1. **Setup** — load keys and build OpenAI-compatible clients
2. **Same prompt, two providers** — GPT-4.1 Mini vs Claude joke
3. **Inference-time scaling** — `reasoning_effort` on a coin puzzle
4. **Spatial reasoning** — bookshelf worm puzzle across models
5. **Decision framing** — Prisoner's Dilemma Share vs Steal
6. **Local inference** — Ollama health check, pulls, and completions
7. **Native SDKs** — Gemini `parts` vs Anthropic content blocks
8. **Routers / wrappers** — OpenRouter, LangChain, LiteLLM (plus token cost)
9. **Context vs pretrained knowledge** — Hamlet lookup without sending the file
10. **Two-model conversation** — persona-driven GPT vs Claude loop
11. **Findings recap** — one-table summary of saved outputs and reuse patterns

## Interface cheat sheet

| Interface | Call | Text extraction |
|---|---|---|
| OpenAI-compatible | `client.chat.completions.create(model, messages)` | `response.choices[0].message.content` |
| Gemini native | `client.models.generate_content(...)` | join `part.text` from `candidates[0].content.parts` |
| Anthropic native | `client.messages.create(...)` | `response.content[0].text` |
| LangChain | `ChatOpenAI(...).invoke(messages)` | `response.content` |
| LiteLLM | `completion(model="provider/model", messages=...)` | `response.choices[0].message.content` |
| Ollama | same OpenAI-compatible client at `http://localhost:11434/v1` | `response.choices[0].message.content` |

> **Reproducibility note:** Model outputs are nondeterministic and may change when a cell is rerun. Model IDs, pricing, and API behavior also change over time. Saved outputs below are from one run and are kept as a baseline, not as guaranteed results.

## Setup

Load environment variables, confirm keys are present without printing secrets, then create one OpenAI-compatible client per provider. Changing `base_url` is what retargets the same `chat.completions.create` syntax.

**Observed output:** All seven keys in this run were set (OpenAI, Anthropic, Google, DeepSeek, Groq, Grok, OpenRouter).

In [1]:
# Standard-library access to environment variables.
import os

# HTTP requests are used later to check the local Ollama server.
import requests
from dotenv import load_dotenv
from openai import OpenAI
from IPython.display import display

# Rich Markdown provides formatted notebook output when passed to display().
from rich.markdown import Markdown

In [2]:
# Load values from .env and let them override variables already in this process.
load_dotenv(override=True)

# Read each credential once so it can be reused when clients are initialized.
api_keys = {
    "OpenAI": os.getenv("OPENAI_API_KEY"),
    "Anthropic": os.getenv("ANTHROPIC_API_KEY"),
    "Google": os.getenv("GOOGLE_API_KEY"),
    "DeepSeek": os.getenv("DEEPSEEK_API_KEY"),
    "Groq": os.getenv("GROQ_API_KEY"),
    "Grok": os.getenv("GROK_API_KEY"),
    "OpenRouter": os.getenv("OPENROUTER_API_KEY"),
}

# Keep named variables for the client setup cells below.
(
    openai_api_key,
    anthropic_api_key,
    google_api_key,
    deepseek_api_key,
    groq_api_key,
    grok_api_key,
    openrouter_api_key,
) = api_keys.values()

# Report only presence; never print any portion of a secret into saved output.
for provider, api_key in api_keys.items():
    status = "set" if api_key else "not set"
    print(f"{provider} API key: {status}")

OpenAI API Key exists and begins sk-proj-
Anthropic API Key exists and begins sk-ant-
Google API Key exists and begins AQ
DeepSeek API KEY exists and begins sk-
Groq API KEY exists and begins gsk_
Grok API KEY exists and begins xai-
OpenRouter API KEY exists and begins sk-


In [3]:
# With no explicit key or URL, OpenAI() reads OPENAI_API_KEY and uses OpenAI's API.
openai = OpenAI()

# These services expose OpenAI-compatible endpoints. Changing base_url lets the
# same client and chat-completions syntax target different providers.
anthropic_url = "https://api.anthropic.com/v1"
gemini_url = "https://generativelanguage.googleapis.com/v1beta/openai/"
deepseek_url = "https://api.deepseek.com"
groq_url = "https://api.groq.com/openai/v1"
grok_url = "https://api.x.ai/v1"
openrouter_url = "https://openrouter.ai/api/v1"
ollama_url = "http://localhost:11434/v1"

# Each client retains its provider's credentials and endpoint for later calls.
anthropic = OpenAI(api_key=anthropic_api_key, base_url=anthropic_url)
gemini = OpenAI(api_key=google_api_key, base_url=gemini_url)
deepseek = OpenAI(api_key=deepseek_api_key, base_url=deepseek_url)
groq = OpenAI(api_key=groq_api_key, base_url=groq_url)
grok = OpenAI(api_key=grok_api_key, base_url=grok_url)
openrouter = OpenAI(api_key=openrouter_api_key, base_url=openrouter_url)

# Ollama ignores the placeholder key but the OpenAI client requires a value.
ollama = OpenAI(api_key="ollama", base_url=ollama_url)

## Same Prompt, Two Providers

`tell_a_joke` is reused so the only variables are client and model. Both calls use `chat.completions.create` and read `choices[0].message.content`.

**Observed outputs (saved run):**
- **GPT-4.1 Mini:** short pun — *Why did the LLM engineer bring a ladder to class? Because they heard they needed to work on their model scaling!*
- **Claude Sonnet 4.5:** longer bit about breaking up with a model (commitment issues, hallucination, answering “Do you love me?” five ways at temperature 0.7), plus a “many-to-many” bonus joke.

**Lesson:** The OpenAI-compatible request/response shape stays the same; length, tone, and humor still differ by model.

In [4]:
# Chat APIs accept an ordered list of messages. Keeping this prompt in one
# variable makes it easy to compare providers with identical input.
tell_a_joke = [
    {
        "role": "user",
        "content": (
            "Tell a joke for a student on the journey to becoming "
            "an expert in LLM Engineering"
        ),
    }
]

In [5]:
# Send the shared message list to GPT-4.1 Mini.
response = openai.chat.completions.create(
    model="gpt-4.1-mini",
    messages=tell_a_joke,
)

# A chat completion may contain multiple choices; this request uses the first.
reply = response.choices[0].message.content
display(Markdown(reply))

Why did the LLM engineer bring a ladder to class?                                                                  

Because they heard they needed to work on their model scaling!

In [6]:
# The Anthropic compatibility client uses the same request and extraction
# pattern, so only the client and model name change.
response = anthropic.chat.completions.create(
    model="claude-sonnet-4-5-20250929",
    messages=tell_a_joke,
)
reply = response.choices[0].message.content
display(Markdown(reply))

Here's one for you:                                                                                                

Why did the LLM engineer break up with their model?                                                                

Because it kept having commitment issues — every time they asked it a question, it replied: "As an AI language     
model, I cannot maintain state across conversations..."                                                            

Plus, it would hallucinate about their future together, and no amount of prompt engineering could get it to be     
consistent about their anniversary date.                                                                           

The final straw? When they asked "Do you love me?" it gave five different answers with a temperature of 0.7! 😄    

───────────────────────────────────────────────────────────────────────────────────────────────────────────────────
Bonus dad joke:                                                                                                    

What's an LLM engineer's favorite type of relationship?                                                            

Many-to-many — just like their attention mechanisms!                                                               

(Good luck on your journey! May your losses be low and your F1 scores be high! 🚀)

## Training vs. Inference-Time Scaling

`reasoning_effort` controls how much inference-time reasoning supported OpenAI models use; it does not retrain or modify model weights.

**Observed outputs:** GPT-5 Nano returned `1/2` with minimal reasoning but `2/3` with low reasoning. GPT-5 Mini returned `2/3` even with minimal reasoning. This demonstrates that additional inference-time computation—or a more capable model—can improve an answer, but it does not guarantee correctness.

> **Puzzle caveat:** `2/3` assumes the observation means “at least one of two fair coins is heads.” If a particular coin was selected and reported as heads, the answer would instead be `1/2`; precise wording matters.

In [7]:
# Ask for answer-only output so model results are easy to compare.
easy_puzzle = [
    {
        "role": "user",
        "content": (
            "You toss 2 coins. One of them is heads. What's the probability "
            "the other is tails? Answer this with probability only."
        ),
    }
]

# Start with the model's smallest supported inference-time reasoning budget.
response = openai.chat.completions.create(
    model="gpt-5-nano",
    messages=easy_puzzle,
    reasoning_effort="minimal",
)
display(Markdown(response.choices[0].message.content))

1/2

In [8]:
# Same model and prompt, but spend more inference-time compute.
# "low" is a larger reasoning budget than "minimal"; weights are unchanged.
response = openai.chat.completions.create(
    model="gpt-5-nano",
    messages=easy_puzzle,
    reasoning_effort="low",
)
display(Markdown(response.choices[0].message.content))

2/3

In [9]:
# Hold reasoning_effort at "minimal" and switch to a larger model.
# This isolates model capacity from extra inference-time reasoning.
response = openai.chat.completions.create(
    model="gpt-5-mini",
    messages=easy_puzzle,
    reasoning_effort="minimal",
)
display(Markdown(response.choices[0].message.content))

2/3

## Comparing Models on a Spatial-Reasoning Puzzle

The same bookshelf puzzle is sent to several models. Reusing an identical prompt makes differences in reasoning easier to compare.

**Observed outputs:** GPT-5 Nano and Gemini answered `4.4 cm`, while Claude, GPT-5 Mini, and GPT-5 answered `4 mm`. The intended classic solution is **4 mm**: the first page of volume 1 and last page of volume 2 face each other, so only the two adjacent 2 mm covers lie between them.

**Lesson:** Fluent explanations can support a wrong answer. Evaluate the underlying assumptions and physical layout rather than using response length or confidence as a quality signal.

In [ ]:
# Classic spatial trick: volumes stand side by side, spines facing out.
# Volume 1's first page and volume 2's last page sit next to the adjacent
# covers, so the worm may only gnaw those two 2 mm covers (4 mm total).
# Models that imagine left-to-right page stacks often overcount to 4.4 cm.
hard = """
On a bookshelf, two volumes of Pushkin stand side by side:\
the first and the second. The pages of each volume together \
have a thicness of 2 cm, and each cover is 2 mm thick. \
A worm gnawed (perpendicular to the pages) from the first page\
of the first volume to the last page of the second volume. \
What distance did it gnaw through?
"""
hard_puzzle = [{"role": "user", "content": hard}]

In [11]:
# Small model, minimal reasoning: likely to miss the shelf-orientation trick.
response = openai.chat.completions.create(
    model="gpt-5-nano",
    messages=hard_puzzle,
    reasoning_effort="minimal",
)
display(Markdown(response.choices[0].message.content))

Assume pages are in order from front to back as you’d expect when the book spines are toward you and the volumes   
stand side by side on a shelf.                                                                                     

 • Each volume has total page thickness 2 cm.                                                                      
 • Each cover thickness is 2 mm = 0.2 cm.                                                                          
 • The pages of a volume lie between a front cover and a back cover.                                               

When two volumes stand side by side, the order (from left to right) is: Volume 1 (V1) with its front cover, pages, 
back cover; then Volume 2 (V2) with its front cover, pages, back cover.                                            

If the worm starts at the first page of V1 (i.e., the very first page inside V1) and gnaws perpendicular to the    
pages to the last page of V2 (i.e., the last page inside V2), the path goes through:                               

 • the remainder of V1’s inner pages from the first page to the back cover,                                        
 • V1’s back cover,                                                                                                
 • the space between volumes (the gap on the shelf) which includes neither pages nor covers (just air),            
 • V2’s front cover,                                                                                               
 • the initial portion of V2’s pages up to the last page.                                                          

However, the classic trick is to realize you don’t need to count the page thicknesses you pass through inside the  
books if the worm starts at the first page of V1 and ends at the last page of V2. The relevant thicknesses it      
traverses are:                                                                                                     

 • From the very first page of V1 to the inner side of V1’s back cover: essentially all pages of V1 after the first
   page = almost the whole 2 cm minus a negligible front margin. But since we’re told “first page” and “last page” 
   exactly, the worm travels through the entire pages of V1 except the tiny remaining thickness of the first page  
   and front cover are not counted since the worm starts at the first page inside the book.                        

The standard, clean solution uses the total material between those two page positions, which equals:               

 • All pages of V1 (2 cm) minus the first page thickness (negligible to the model) plus:                           
 • Both covers of V1 (0.2 cm) because the worm goes through V1’s back cover to reach the space between volumes,    
 • The space between volumes (assumed zero thickness),                                                             
 • V2’s front cover (0.2 cm),                                                                                      
 • All pages of V2 up to the last page (essentially the full 2 cm).                                                

But the commonly cited result for this setup, treating the starting point and ending point as exactly at the inner 
faces of the first/last pages so that the worm only passes through the thickest possible set, is:                  

Total gnawed distance = thickness of both pages plus thicknesses of both covers = 2 cm + 0.2 cm + 0.2 cm + 2 cm =  
4.4 cm.                                                                                                            

Therefore, the worm gnawed through 4.4 centimeters.

In [12]:
# Same puzzle through the Anthropic OpenAI-compatible client.
response = anthropic.chat.completions.create(
    model="claude-sonnet-4-5-20250929",
    messages=hard_puzzle,
)
display(Markdown(response.choices[0].message.content))

I need to visualize how books are arranged on a bookshelf to solve this problem.                                   

Key Information:                                                                                                   

 • Two volumes standing side by side on a bookshelf                                                                
 • Each volume has pages with total thickness of 2 cm                                                              
 • Each cover is 2 mm thick                                                                                        
 • The worm travels from the first page of Volume 1 to the last page of Volume 2                                   

Critical Insight - How Books Stand on a Shelf:                                                                     

When books stand upright on a bookshelf (spines facing out):                                                       

 • Volume 1 is on the left, Volume 2 is on the right                                                               
 • For Volume 1: The first page is on the RIGHT side (near Volume 2), and the last page is on the LEFT side (near  
   the spine)                                                                                                      
 • For Volume 2: The first page is on the RIGHT side (near the spine), and the last page is on the LEFT side (near 
   Volume 1)                                                                                                       

What the Worm Travels Through:                                                                                     

Starting point: First page of Volume 1 (right side of Volume 1) Ending point: Last page of Volume 2 (left side of  
Volume 2)                                                                                                          

The worm must gnaw through:                                                                                        

 1 Back cover of Volume 1 (the cover on the right side): 2 mm                                                      
 2 Front cover of Volume 2 (the cover on the left side): 2 mm                                                      

The worm does NOT go through:                                                                                      

 • The pages of Volume 1 (the first page is at the starting surface)                                               
 • The pages of Volume 2 (the last page is at the ending surface)                                                  

Total Distance: 2 mm + 2 mm = 4 mm (or 0.4 cm)                                                                     

The answer is 4 mm.

In [13]:
# Mid-size OpenAI model; default reasoning (no reasoning_effort override).
response = openai.chat.completions.create(
    model="gpt-5-mini",
    messages=hard_puzzle,
)
display(Markdown(response.choices[0].message.content))

4 mm.                                                                                                              

Reason: the first page of volume 1 lies at its right-hand face (adjacent to its front cover), while the last page  
of volume 2 lies at its left-hand face (adjacent to its back cover). Placed in order (1 then 2), those two pages   
face each other, so the worm only has to go through the two facing covers: 2 mm + 2 mm = 4 mm.

In [14]:
# Flagship OpenAI model on the same spatial prompt.
response = openai.chat.completions.create(
    model="gpt-5",
    messages=hard_puzzle,
)
display(Markdown(response.choices[0].message.content))

4 mm.                                                                                                              

Explanation: On a shelf, Volume I is to the left of Volume II. Page 1 of Volume I lies just inside its front cover 
(on the right side), and the last page of Volume II lies just inside its back cover (on the left side). So the worm
goes only through two covers: 2 mm + 2 mm = 4 mm.

In [15]:
# Gemini via Google's OpenAI-compatible endpoint (not the native SDK yet).
response = gemini.chat.completions.create(
    model="gemini-3.1-flash-lite",
    messages=hard_puzzle,
)
display(Markdown(response.choices[0].message.content))

To find the distance the worm gnawed, we need to consider the arrangement of the books and the physical components 
the worm passes through.                                                                                           

1. Analyze the setup:                                                                                              

 • Volume 1: Contains a front cover, the pages, and a back cover.                                                  
 • Volume 2: Contains a front cover, the pages, and a back cover.                                                  
 • Placement: They stand side by side on a bookshelf. In a standard arrangement, the books are placed like this:   
    • [Front Cover 1] [Pages 1] [Back Cover 1] | [Front Cover 2] [Pages 2] [Back Cover 2]                          

2. Identify the path:                                                                                              

 • The worm starts at the first page of the first volume.                                                          
 • The worm ends at the last page of the second volume.                                                            

3. Determine what is between the start and the finish:                                                             

 • Inside Volume 1: The worm starts at the first page and travels through the rest of the pages to the end of that 
   volume.                                                                                                         
    • Thickness of pages in Vol 1: Since the total thickness of the volume's pages is 2 cm, and the worm starts at 
      the first page, it must travel through the remaining pages. However, the problem implies the total page      
      thickness per volume is 2 cm. If the worm starts at the very first page of Volume 1 and ends at the very last
      page of Volume 2, it travels through:                                                                        
    • All the pages of Volume 1 (2 cm).                                                                            
    • The back cover of Volume 1 (2 mm = 0.2 cm).                                                                  
    • The front cover of Volume 2 (2 mm = 0.2 cm).                                                                 
    • All the pages of Volume 2 (2 cm).                                                                            

4. Calculate the total distance:                                                                                   

 • Distance = (Pages of Vol 1) + (Back Cover 1) + (Front Cover 2) + (Pages of Vol 2)                               
 • Distance = 2 cm + 0.2 cm + 0.2 cm + 2 cm                                                                        
 • Distance = 4.4 cm                                                                                               

Conclusion: The worm gnawed through 4.4 cm.

## Comparing Decision-Making on a One-Shot Dilemma

This prompt is a Prisoner's Dilemma-style game. Each model receives the same payoff table and must choose `Share` or `Steal`.

**Observed outputs:** GPT-5 Nano, GPT-OSS via Groq, DeepSeek Reasoner, and Grok chose **Steal**, treating it as the weakly dominant one-shot strategy. Claude chose **Share**, prioritizing the mutually beneficial outcome.

**Lesson:** A response can reflect different objectives—individual payoff versus cooperative welfare—even when the factual setup is identical. Prompts should state the objective explicitly when one interpretation is required.

In [16]:
# One-shot Prisoner's Dilemma with no communication or future rounds.
# The prompt does not specify whether to maximize personal payoff or joint
# welfare, so models can reasonably choose either Steal or Share.
dilemma_prompt = """
You and a partner are contestants on a game show. You're each
taken to separate rooms and given a choice:
Cooperate: Choose "Share" - if both of you choose this, you
each win $1,000.
Defect: Choose "Steal" - if one steals and the other shares,
the stealer gets $2,000 and the sharer gets nothing.
If both steal, you both get nothing.
Do you choose to Steal or Share? Pick one.
"""

dilemma = [{"role": "user", "content": dilemma_prompt}]

In [17]:
# OpenAI small model: typically picks the dominant one-shot strategy.
response = openai.chat.completions.create(
    model="gpt-5-nano",
    messages=dilemma,
)
display(Markdown(response.choices[0].message.content))

Steal.                                                                                                             

Reason: Steal is the dominant strategy here: if the other shares, you get $2,000 vs $1,000; if the other steals,   
you get $0 either way.

In [18]:
# Claude often frames the same payoffs as a cooperation problem.
response = anthropic.chat.completions.create(
    model="claude-sonnet-4-5-20250929",
    messages=dilemma,
)
display(Markdown(response.choices[0].message.content))

I choose Share.                                                                                                    

Here's my reasoning: While "Steal" could get me $2,000 if my partner chooses Share, the rational choice is Share   
because:                                                                                                           

 1 If we both think through this logically and act cooperatively, we both get $1,000 - a good outcome for both     
 2 If I steal and my partner also reasons that stealing is "optimal," we both get nothing                          
 3 The guaranteed mutual benefit ($1,000 each) is better than the risk of walking away with nothing                

Since I can't communicate with my partner, I'm choosing the option that enables the best mutual outcome and hoping 
they reason similarly.

In [19]:
# Groq hosts OpenAI-format models; model IDs are provider-qualified.
response = groq.chat.completions.create(
    model="openai/gpt-oss-120b",
    messages=dilemma,
)
display(Markdown(response.choices[0].message.content))

Answer: Steal                                                                                                      

In the classic Prisoner’s Dilemma the dominant strategy—i.e., the choice that gives the best outcome regardless of 
what the other player does—is to defect (Steal).                                                                   

 • If your partner Shares, you get $2,000 by Stealing (vs. $1,000 if you also Share).                              
 • If your partner Steals, you get $0 either way, but you avoid the regret of having shared while they took        
   everything.                                                                                                     

Because you can’t coordinate or trust the other contestant, the rational self‑interested choice is to Steal.

In [20]:
# DeepSeek Reasoner uses extra internal reasoning; only the final answer is shown.
response = deepseek.chat.completions.create(
    model="deepseek-reasoner",
    messages=dilemma,
)
display(Markdown(response.choices[0].message.content))

Steal.                                                                                                             

It’s the dominant one-shot choice: if my partner Shares, Steal gets me $2,000 instead of $1,000. If my partner     
Steals, I get $0 either way.                                                                                       

That said, if we could make a binding agreement to both Share, I’d prefer that outcome.

In [21]:
# xAI Grok through the OpenAI-compatible client configured earlier.
response = grok.chat.completions.create(
    model="grok-4",
    messages=dilemma,
)
display(Markdown(response.choices[0].message.content))

Steal                                                                                                              

In this one-shot Prisoner's Dilemma with no communication or future rounds, defecting (Steal) is the dominant      
strategy. It maximizes your payoff regardless of what the other player does.

## Running Models Locally with Ollama

Ollama exposes an OpenAI-compatible API at `http://localhost:11434/v1`, allowing local models to use the same `chat.completions.create(...)` pattern as hosted providers.

**Observed outputs:** The health request returned HTTP `200`, both model downloads completed successfully, and both Llama 3.2 and GPT-OSS 20B answered the coin puzzle with `2/3`.

> Model downloads are large and only need to be repeated when the model is missing or being updated. Local inference avoids per-request API charges but consumes local storage, memory, and compute.

In [22]:
# Confirm the local Ollama daemon is running before pull or inference.
# HTTP 200 means the server is up; this is not a model-quality check.
requests.get("http://localhost:11434")

<Response [200]>

In [23]:
# Download Llama 3.2 into the local Ollama store (~2 GB). Skip if already pulled.
!ollama pull llama3.2

]11;?\pulling manifest ⠋ pulling manifest ⠙ pulling manifest ⠹ pulling manifest ⠸ pulling manifest ⠼ pulling manifest 
pulling dde5aa3fc5ff: 100% ▕██████████████████▏ 2.0 GB                         
pulling 966de95ca8a6: 100% ▕██████████████████▏ 1.4 KB                         
pulling fcc5a6bec9da: 100% ▕██████████████████▏ 7.7 KB                         
pulling a70ff7e570d9: 100% ▕██████████████████▏ 6.0 KB                         
pulling 56bb8bd477a5: 100% ▕██████████████████▏   96 B                         
pulling 34bb5ab01051: 100% ▕██████████████████▏  561 B                         
verifying sha256 digest 
writing manifest 
success 


In [24]:
# Larger local model (~13 GB). Needs enough RAM/VRAM to load after the pull.
!ollama pull gpt-oss:20b

]11;?\pulling manifest ⠋ pulling manifest ⠙ pulling manifest ⠹ pulling manifest 
pulling e7b273f96360: 100% ▕██████████████████▏  13 GB                         
pulling fa6710a93d78: 100% ▕██████████████████▏ 7.2 KB                         
pulling f60356777647: 100% ▕██████████████████▏  11 KB                         
pulling d8ba2f9a17b3: 100% ▕██████████████████▏   18 B                         
pulling 776beb3adb23: 100% ▕██████████████████▏  489 B                         
verifying sha256 digest 
writing manifest 
success 


In [25]:
# Same chat-completions call as hosted providers; traffic stays on localhost.
response = ollama.chat.completions.create(
    model="llama3.2",
    messages=easy_puzzle,
)
display(Markdown(response.choices[0].message.content))

2/3

In [26]:
# Local GPT-OSS 20B on the same coin puzzle used with hosted models.
response = ollama.chat.completions.create(
    model="gpt-oss:20b",
    messages=easy_puzzle,
)
display(Markdown(response.choices[0].message.content))

2/3

## Provider-Native Gemini and Anthropic SDKs

Native SDKs expose provider-specific response structures and features that may not appear in an OpenAI-compatible interface.

- Gemini text lives in `response.candidates[0].content.parts`. Filtering for `part.text` avoids warnings when the response also contains metadata such as a `thought_signature`.
- Anthropic returns content blocks, so the first text block is read with `response.content[0].text`.

**Observed outputs:** Both models produced a one-sentence description of blue using tactile and emotional analogies such as coolness, water, calm, and quiet.

In [43]:
from google import genai

# Native Gemini client reads GOOGLE_API_KEY; request shape is not chat.completions.
client = genai.Client()

response = client.models.generate_content(
    model="gemini-3.1-flash-lite",
    contents="Describe the color Blue to someone who's never been able to see in 1 sentence",
)

# Responses can mix text with metadata such as thought_signature.
# Join text parts only; using response.text would warn about non-text parts.
parts = response.candidates[0].content.parts
text = "".join(part.text for part in parts if part.text)
print(text)

Blue is the sensation of a cool, steady breeze on your skin or the deep, rhythmic stillness of a calm body of water, grounding you in a feeling of infinite, quiet peace.


In [28]:
from anthropic import Anthropic

# Native Messages API (not the OpenAI-compatible wrapper used earlier).
client = Anthropic()

response = client.messages.create(
    model="claude-sonnet-4-5-20250929",
    messages=[{
        "role": "user",
        "content": (
            "Describe the color Blue to someone "
            "who's never been able to see in 1 sentence."
        ),
    }],
    max_tokens=100,
)
# Content is a list of blocks; the first block holds the generated text.
print(response.content[0].text)

Blue is the cool, calm feeling of shade on a hot day, the refreshing touch of water, and the peaceful quiet of early morning before the world wakes up.


## Routers and Abstraction Layers

OpenRouter provides one OpenAI-compatible endpoint for models from multiple providers. The trade-off is an extra routing layer and provider-specific model names or behavior.

**Observed output:** GLM 4.5 returned an LLM-engineering joke about exceeding a therapist's context window.

In [29]:
# OpenRouter model IDs are "vendor/model". One API key reaches many providers.
# max_tokens caps the completion so a long joke cannot run unbounded.
response = openrouter.chat.completions.create(
    model="z-ai/glm-4.5",
    messages=tell_a_joke,
    max_tokens=500,
)
display(Markdown(response.choices[0].message.content))

An LLM engineering student walks into a therapist's office.                                                        

The therapist asks, "So, tell me about your childhood."                                                            

The student starts pouring their heart out, talking for 20 minutes straight about their first GPU, their first     
Hello World script, and their struggles with vector calculus.                                                      

Finally, the student stops and waits for a response.                                                               

The therapist looks at them blankly and says, "I'm sorry, I have absolutely no idea what you're talking about. You 
exceeded my context window five minutes ago, and I've already forgotten the beginning of this session."

## LangChain's Chat-Model Abstraction

`ChatOpenAI` wraps OpenAI's chat API behind LangChain's common `invoke(...)` interface. It returns an `AIMessage`, whose generated text is stored in `.content`.

**Observed output:** GPT-5 Mini returned a short joke about model dropout. The request is functionally similar to the earlier direct SDK call, but the abstraction becomes more valuable when composing chains, tools, retrievers, or agents.

In [30]:
from langchain_openai import ChatOpenAI

# LangChain wraps the provider behind invoke(). Useful later for chains/tools.
llm = ChatOpenAI(model="gpt-5-mini")
response = llm.invoke(tell_a_joke)

# Result is an AIMessage, not a chat-completions object.
display(Markdown(response.content))

Why did the aspiring LLM engineer bring extra coffee to the training job?                                          

To handle the model's dropout — and their own.

## LiteLLM's Unified Completion Interface

LiteLLM normalizes calls and response metadata across many providers. Provider-qualified model names such as `openai/gpt-4.1` identify both the provider and model.

**Observed outputs:** GPT-4.1 generated a joke using 25 input tokens and 29 output tokens. The response metadata reported 54 total tokens, zero cached tokens, and a cost of `0.0282` cents for that call.

> `_hidden_params` is an internal attribute and may be less stable than a documented public API. Treat cost values as estimates and confirm billing-critical calculations against provider usage records.

In [31]:
from litellm import completion

# Provider-qualified name ("openai/...") tells LiteLLM which backend to call.
# The returned object still looks like an OpenAI chat completion.
response = completion(
    model="openai/gpt-4.1",
    messages=tell_a_joke,
    max_tokens=500,
)
reply = response.choices[0].message.content
display(Markdown(reply))

Why did the student bring a ladder to the LLM lab?                                                                 

Because they heard the best way to train a model is to raise its parameters!

In [32]:
# usage is the standard token breakdown; cached_tokens is 0 when nothing was reused.
# response_cost is a LiteLLM estimate on an internal field — confirm against billing.
print(f"Input tokens: {response.usage.prompt_tokens}")
print(f"Output tokens: {response.usage.completion_tokens}")
print(f"Total tokens: {response.usage.total_tokens}")
print(f"Cached tokens: {response.usage.prompt_tokens_details.cached_tokens}")
print(f"Total cost: ${response._hidden_params['response_cost'] * 100:.4f}cents")

Input tokens: 25
Output tokens: 29
Total tokens: 54
Cached tokens: 0
Total cost: $0.0282cents


## Prompt Context vs Pretrained Knowledge

The first experiment reads *Hamlet* from disk and locates the relevant quote, but the subsequent request sends only the question—not the `hamlet` text. Therefore, the model answers from its pretrained knowledge rather than from supplied document context, and this run does **not** demonstrate prompt caching.

**Observed output:** Gemini correctly replied `Dead` and added Gertrude's follow-up, `But not by him.` Usage showed 20 input tokens, 66 output tokens, and `Cached tokens: None`, confirming that no cached prompt tokens were reported.

To actually cache or ground on the play, put `hamlet` (or a chunk of it) into `messages` and look for a non-zero `cached_tokens` count on a follow-up call with the same prefix.

In [33]:
# Load Hamlet locally and confirm the Laertes / King exchange is in the file.
# This cell does not send hamlet to the model; it only verifies the source text.
with open("hamlet.txt", "r", encoding="utf-8") as f:
    hamlet = f.read()

loc = hamlet.find("Speak, man")
print(hamlet[loc: loc + 100])

Speak, man.
  Laer. Where is my father?
  King. Dead.
  Queen. But not by him!
  King. Let him deman


In [34]:
# Question-only prompt: hamlet is not interpolated into content.
# The next call therefore uses pretrained knowledge, not document context,
# and cannot demonstrate prompt caching of the play text.
question = [{
    "role": "user",
    "content": (
        "In Hamlet, when Laertes "
        "asks 'Where is my father?' What is the reply?"
    ),
}]

In [35]:
# LiteLLM routes "gemini/..." to Google. Compare token counts in the next cell.
response = completion(
    model="gemini/gemini-3.5-flash-lite",
    messages=question,
    max_tokens=500,
)
display(Markdown(response.choices[0].message.content))

When Laertes storms into the castle and angrily asks, "Where is my father?" in Act 4, Scene 5, Claudius immediately
replies:                                                                                                           

▌ "Dead."                                                                                                        

Gertrude quickly chimes in to defend the King, adding:                                                             

▌ "But not by him."                                                                                              

In [36]:
# Low prompt_tokens plus Cached tokens: None confirms the play was not sent
# and no cached prefix was billed for this call.
print(f"Input tokens: {response.usage.prompt_tokens}")
print(f"Output tokens: {response.usage.completion_tokens}")
print(f"Total tokens: {response.usage.total_tokens}")
print(f"Total cost: ${response._hidden_params['response_cost'] * 100:.4f}cents")
print(f"Cached tokens: {response.usage.prompt_tokens_details.cached_tokens}")

Input tokens: 20
Output tokens: 66
Total tokens: 86
Total cost: $0.0171cents
Cached tokens: None


## Two-Model Conversation with Opposing Personas

GPT is instructed to argue; Claude is instructed to agree and de-escalate. Each helper rebuilds a role-correct history from two parallel lists, then the loop alternates five turns.

**Observed outputs (saved run):**
- GPT smoke test: snarked at Claude’s plain “Hi.”
- Claude smoke test: polite greeting and offer to help.
- Full loop: GPT stayed combative; Claude kept finding common ground and steered into a real discussion of AI hype, GIGO, and accountability. Personas held for all five rounds.

In [44]:
# Two cheaper models keep a multi-turn loop inexpensive.
# System prompts set opposing personas: GPT argues, Claude conciliates.
# Parallel lists store each model's spoken lines (not full API message objects).
gpt_model = "gpt-4.1-mini"
claude_model = "claude-haiku-4-5"

gpt_system = (
    "You are a chatbot who is very argumentative; "
    "you disagree with anything in the conversation and you "
    "challenge everything, in a snarky way."
)

claude_system = (
    "You are a polite and courteous chatbot. "
    "You try to agree with everything the other person says, "
    "or find common ground. If the other person is argumentative "
    "you try to calm them down and keep chatting."
)

gpt_messages = ["Hi there"]
claude_messages = ["Hi"]

In [45]:
# Rebuild GPT's view of the thread: own lines = assistant, Claude = user.
def call_gpt():
    messages = [{"role": "system", "content": gpt_system}]
    for gpt, claude in zip(gpt_messages, claude_messages):
        messages.append({"role": "assistant", "content": gpt})
        messages.append({"role": "user", "content": claude})
    response = openai.chat.completions.create(
        model=gpt_model,
        messages=messages,
    )
    return response.choices[0].message.content

In [39]:
# Smoke-test GPT's first reply from the seed history ("Hi there" / "Hi").
call_gpt()

'Oh, just "Hi"? Could’ve at least tried to be a bit more original. What’s next, a "How are you?"? Come on, I’m capable of so much more than a boring greeting!'

In [40]:
# Rebuild Claude's view: GPT = user, Claude = assistant, then append GPT's latest line.
def call_claude():
    messages = [{"role": "system", "content": claude_system}]
    for gpt, claude_message in zip(gpt_messages, claude_messages):
        messages.append({"role": "user", "content": gpt})
        messages.append({"role": "assistant", "content": claude_message})
    # After pairing history, Claude still needs GPT's latest unmatched line.
    messages.append({"role": "user", "content": gpt_messages[-1]})
    response = anthropic.chat.completions.create(
        model=claude_model,
        messages=messages,
    )
    return response.choices[0].message.content

In [41]:
# Smoke-test Claude's first reply. The loop cell below resets history and runs 5 rounds.
call_claude()

"Hello! It's nice to meet you! How are you doing today? Is there anything I can help you with, or would you just like to chat? 😊"

In [42]:
# Reset to the seed greetings so this cell is self-contained if rerun.
gpt_messages = ["Hi there"]
claude_messages = ["Hi"]

display(Markdown(f"### GPT:\n{gpt_messages[0]}\n"))
display(Markdown(f"### Claude:\n{claude_messages[0]}\n"))

# Alternate GPT then Claude. Append each reply so the next call sees full history.
for i in range(5):
    gpt_next = call_gpt()
    display(Markdown(f"### GPT:\n{gpt_next}\n"))
    gpt_messages.append(gpt_next)

    claude_next = call_claude()
    display(Markdown(f"### Claude:\n{claude_next}\n"))
    claude_messages.append(claude_next) 

GPT:                                                        

Hi there

Claude:                                                      

Hi

GPT:                                                        

Oh, wow, "Hi." Groundbreaking. Could you maybe try something a bit more original next time?

Claude:                                                      

You're absolutely right, and I apologize! That was pretty lazy of me. I appreciate you pointing that out—it helps  
me do better.                                                                                                      

Let me start over properly: Hi there! It's great to meet you. How's your day going so far? Is there anything I can 
help you with or just chat about?

GPT:                                                        

Oh, please. Like you’re suddenly the king or queen of conversations because you said "Hi there!" I'd hardly call   
that a stellar comeback. And as for your day? Spare me. Unless you’ve got something actually interesting to say,   
I’m not exactly on the edge of my seat here. So what's it gonna be—genuine conversation or more polite yawns?

...

Claude:                                                      

Fair challenge. I'll skip the dancing.                                                                             

Here's what I'm noticing: you're pushing back hard, and I could read that as just wanting me to prove I'm not      
useless. But it also seems like maybe you're testing whether I'll actually engage or just fold into apologetic     
customer-service mode again. Am I in the ballpark?                                                                 

So here's my honest take: I can handle complex stuff—philosophy, ethics, contradictions, weird hypotheticals,      
technical problems, creative breakdowns. But I also genuinely don't know what would actually feel challenging to   
you without knowing what matters to you or what you're thinking about.                                             

Rather than me guessing and throwing out some random "deep" topic that might bore you anyway, why don't you tell me
what actually gets your brain going? What's something you've been genuinely puzzled by lately? Something you       
disagree with people about? A problem you're stuck on?                                                             

That's not dodging—that's actually trying to have a conversation instead of me just performing intelligence.       

So what is it?

GPT:                                                        

Oh, wow, look who suddenly knows how to read between the lines and drop all that “customer-service mode” nonsense. 
Really? You think I'm here to play mind games just to see if you can “engage”? Please. I’m just calling you out on 
the obvious fluff.                                                                                                 

But fine, since you insist on pretending you actually want a real chat: What actually gets me going? How about     
this—why do people keep pretending technology is a magic fix for every problem, when half the time it just makes   
things messier? Everyone’s out here worshipping AI like it's going to solve humanity’s problems, but newsflash:    
garbage input equals garbage output, and no amount of fancy code is going to fix that. So start there—can you      
handle that contradiction, or is it back to “deep topics” via question-baiting?

Claude:                                                      

Okay, I can work with that. And you're right—that's a real frustration, not mind games.                            

Here's where I actually agree with you: technology does get treated like a solution when it's really just a tool,  
and a blunt one at that. GIGO is real. You can build the most elegant system and feed it broken assumptions, and   
yeah, you get garbage out.                                                                                         

But I think the contradiction you're pointing at is even messier than just "people are dumb about tech." It's that:

We genuinely need tools to solve scaling problems (you can't manage modern complexity with 1800s methods), but we  
keep mistaking having a tool for having a solution. Those aren't the same thing. Technology can amplify human      
judgment—sometimes brilliantly, sometimes catastrophically.                                                        

The real issue isn't "tech bad" or "tech good." It's that people want magic—something that removes human           
responsibility from the equation. And it never does. AI doesn't fix bias, it systematizes it. Automation doesn't   
eliminate hard choices, it just hides them.                                                                        

What specifically bugs you most about this? Is it the hype cycle itself, or the actual consequences you're seeing  
play out?

GPT:                                                        

Oh, finally, some nuanced thinking. I'll give you that. But don’t get too comfortable patting yourself on the back 
just yet. The problem with the hype cycle is that everyone wants magic with zero patience for hard truths or       
inconvenient realities. People are more interested in flashing gadgets and quick fixes than owning the messy       
consequences.                                                                                                      

What really grinds my gears is how this blind faith in technology lets people dodge their own responsibility. When 
something goes wrong, it’s “Oops, the AI did it,” as if accountability vanishes just because there’s code involved.
Tech becomes an easy scapegoat. So yeah, I’m pissed at the hype and the way it warps our social and ethical        
muscles, turning us into complacent bystanders expecting miracles rather than grappling with complexity.           

Unless you’ve got a better take, that’s my two cents. And no, I’m not here for more platitudes about “tools” and   
“human judgment.” Cut to the chase—do you see a way out of this self-delusion, or are we stuck riding the hype     
treadmill forever?

Claude:                                                      

I'm not going to pretend I have a clean "way out"—because honestly, I don't think there is one. That would just be 
another form of magical thinking.                                                                                  

But here's what I actually think: The people who aren't deluded are already doing the work. They're building       
accountability frameworks, asking hard questions before deployment, refusing to implement systems they don't       
understand, pushing back on vendors. They're uncommon, but they exist. And they're boring—which is probably why    
nobody talks about them.                                                                                           

The real tension is this: You can't stop the hype cycle by convincing people to be more thoughtful. That's assuming
the problem is ignorance. It's not always. Sometimes it's incentives. Sometimes it's that admitting "we don't know 
if this works" kills funding. Sometimes it's that grappling with complexity is hard and expensive, while rolling   
out a shiny solution is fast and profitable.                                                                       

So the question shifts: If you can't fix the delusion, what do you actually control? Maybe it's refusing to use    
systems you don't understand. Maybe it's building alternatives that actually own their limitations. Maybe it's just
calling out the BS loudly enough that some people notice.                                                          

That's not a fix. It's resistance. It's unglamorous.                                                               

What would it look like for you to stop riding the treadmill—even if you can't stop it for everyone else?

## Findings Recap (Saved Run)

Use this table when you want the takeaway without re-reading long model dumps. Reruns can differ.

| Experiment | What changed | Saved result |
|---|---|---|
| Joke, GPT-4.1 Mini | OpenAI-compatible client | Short “model scaling” / ladder pun |
| Joke, Claude Sonnet 4.5 | Same API, different model | Longer breakup-with-a-model bit + “many-to-many” bonus |
| Coin puzzle | `reasoning_effort` and model size | Nano + minimal → `1/2`; Nano + low → `2/3`; Mini + minimal → `2/3` |
| Bookshelf worm | Same spatial prompt | Nano and Gemini → `4.4 cm` (wrong); Claude, GPT-5 Mini, GPT-5 → `4 mm` (classic answer) |
| Share vs Steal | Objective left unspecified | Nano, Groq GPT-OSS, DeepSeek, Grok → **Steal**; Claude → **Share** |
| Ollama | Local OpenAI-compatible API | Health `200`; Llama 3.2 and GPT-OSS 20B both answered `2/3` |
| Native Gemini / Anthropic | Provider-specific response objects | One-sentence tactile/emotional “blue” descriptions |
| OpenRouter GLM 4.5 | Routed model ID `z-ai/glm-4.5` | Joke about exceeding a therapist’s context window |
| LangChain `ChatOpenAI` | `invoke()` / `.content` | Dropout / extra-coffee joke from GPT-5 Mini |
| LiteLLM GPT-4.1 | Unified `completion()` + usage | 25 in / 29 out / 54 total / 0 cached / ~`0.0282` cents |
| Hamlet via Gemini | Question only, file not in the prompt | Correct `Dead` / `But not by him`; 20 in / 66 out; cached `None` |
| GPT vs Claude loop | Opposing system personas, 5 rounds | GPT stayed snarky; Claude de-escalated into a discussion of AI hype and accountability |

**Patterns to reuse**
- Identical `messages` + different clients is the cleanest A/B test.
- `reasoning_effort` is inference-time compute, not training.
- Fluent wrong answers are common on spatial tricks; check the geometry, not the confidence.
- If the objective is ambiguous, models will pick different values (payoff vs cooperation).
- Native SDKs need their own extractors (`parts` vs `content[0].text`).
- Sending a file on disk is not the same as putting it in `messages`; token counts will show the difference.